**GEOG5415M Programming for Spatial Data Science**

# Week 7: Machine Learning in Action

In this practical we will go through a worked example of machine learning in action. We are going to replicate some of the work in the paper:

 - Asher, M., Oswald, Y., & Malleson, N. (2025). Understanding pedestrian dynamics using machine learning with real-time urban sensors. _Environment and Planning B: Urban Analytics and City Science_, 52(8), 1994-2017. DOI:[10.1177/23998083251319058](https://doi.org/10.1177/23998083251319058)

The aim of the work is to build a model that can predict _pedestrian footfall_ for a particular hour, given information about the time, the weather, and the built environment.

If you are interested, the original code, in full, is available from the paper's Github repository: 

 - [github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis)

We will go through the following steps:
 1. Data preparation (including downloading, cleaning and linking)
 1. Data analysis (look for missing data and look at data distributions)
 1. Model the data (including model selection)

In [1]:
import os

# Normal data science libraries 
import geopandas as gpd
import pandas as pd
import seaborn as sns
#from scipy import stats
import numpy as np
import matplotlib.pyplot as plt

# machine learning packages
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# set seaborn plotting theme to white
sns.set_theme(style="white")

# Data Preparation

We need to download, process, clean and merge the following data sources:

 - footfall counts (our target)
 - weather conditions
 - built environment features
 - date information (public holidays, school term times, etc,)

If you are interested, the full code is organised into various notebooks in the original repository's [PreparingData](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis/1.%20PreparingData) directory.

## Downloading and reading data

### Footfall counts

The first dataset we need is the footfall counts -- i.e. the counts of pedestrians who pass Melbourne's footfall cameras every hour. In the paper, we had to find the data on the Melbourne Open Data Portal, download a few different files (one for old counts, one for new ones) and merge them. If you want to see the code to do this in full, have a look at the [Data Preparation Folder](https://github.com/nickmalleson/footfall/tree/main/MelbourneAnalysis/1.%20PreparingData) in the paper's repository. For this practical, we have made the data available with the notebook to save some time, but there is still a lot of cleaning and preparation to do.

The data are stored in the file called "sensor_counts.csv.gz" in the data/week_7 directory. Note that the file has the '.gz' extension. This is short for 'gzip' and it means that the csv file has been compressed using an algorithm called 'gzip' so that it takes up less space. Fortunately pandas makes it really easy to read and write files using the argument `compression='gzip'`. As with other data, we can read it directly from the module's github repository

<font color='orchid'> <b>Run the code below to read the compressed csv file and assign it to a variable called `sensor_counts`</b></font>.

In [2]:
url_to_sensor_data = "https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/sensor_counts.csv.gz"
sensor_counts = pd.read_csv(url_to_sensor_data, compression='gzip')

Have a look at the sensor counts. Each row holds the number of counts from a sensor for a particular hour (the `hourly_counts`) column, as well as some other information. There are quite a lot of rows!

In [3]:
sensor_counts

,Unnamed: 0,datetime,year,month,mdate,day,time,sensor_id,hourly_counts
0,0,2019-11-01 17:00:00,2019,November,1,Friday,17,34,300
1,1,2019-11-01 17:00:00,2019,November,1,Friday,17,39,604
2,2,2019-11-01 17:00:00,2019,November,1,Friday,17,37,216
3,3,2019-11-01 17:00:00,2019,November,1,Friday,17,40,627
4,4,2019-11-01 17:00:00,2019,November,1,Friday,17,36,774
...,...,...,...,...,...,...,...,...,...
5480637,1850385,2024-03-22 15:00:00,2024,March,22,Friday,15,45,1388
5480638,1850388,2023-05-27 13:00:00,2023,May,27,Saturday,13,63,1052
5480639,1850389,2024-03-23 16:00:00,2024,March,23,Saturday,16,50,557
5480640,1850390,2024-04-17 16:00:00,2024,April,17,Wednesday,16,20,478


There is a column that we don't need so lets get rid of it. The easiest way to do this is to use the `sensor_counts.drop()` function. For example, if you wanted to remove a column called 'MyColumn' you could remove it with:
```python
sensor_counts.drop(colums = ['MyColumn'])
```

<font color='orchid'> <b>Write the code below remove the column called 'Unnamed: 0'.</b></font> (Hint: don't foget to reassign the output of `drop()` back to your `sensor_counts` variable)

In [4]:
sensor_counts = sensor_counts.drop(columns = ['Unnamed: 0'])

The next chunk will check that the column removal worked correctly. If it runs without an error then you have done it right.
(Ask someone if you'd like us to explain how it works)

In [5]:
column_still_in_df = [c for c in ['Unnamed: 0'] if c in sensor_counts.columns]
if column_still_in_df:
    raise ValueError(f"These columns are still in the dataset: {column_still_in_df}")
print("Column removed successfully")

Column removed successfully


Now we need to check that the remaining colums that we have store their data correctly. Are they stored using an appropriate data type?

<font color='orchid'> <b>Edit the code below to see what types of data the columns store.</b></font> (hint: you can use the `.info()` function to do this).

In [6]:
sensor_counts.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5480642 entries, 0 to 5480641
Data columns (total 8 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   datetime       object
 1   year           int64 
 2   month          object
 3   mdate          int64 
 4   day            object
 5   time           int64 
 6   sensor_id      int64 
 7   hourly_counts  int64 
dtypes: int64(5), object(3)
memory usage: 334.5+ MB


Most are OK. The months and days are text, so these are correctly represented as `object` types. The numerical data are all whole numbers, so correctly represented as `int`s. 

**But what about the `datetime` column?** That is also being stored as a string  object, which means that pandas wont be able to do reliable date aritmetic (e.g. comparing dates, adding to dates, etc.). 

To fix this we can change the type of the colum to a `datetime` object using the `pd.to_datetime()` function. Have a look at the [documentation](https://pandas.pydata.org/pandas-docs/stable/reference/api/pandas.to_datetime.html). It tells as that the only 'positional' (required) argument is a Series (the `arg` parameter). Often we also need to provide the `format` argument to tell it how to convert the string to a date, but in this case our string date is formatted in a standard way so the function can work out what to do.

If we had a string column called `my_date` in the dataframe called `df` we could convert it like this:
```python
df['my_date'] = pd.to_datetime['my_date']
```

<font color='orchid'> <b>Write the code below to convert the column called `datetime` column to a proper 'datetime' object</b></font> 

In [7]:
sensor_counts['datetime'] = pd.to_datetime(sensor_counts['datetime'])

This next chunk checks that the conversion worked ok

In [8]:
if not pd.api.types.is_datetime64_any_dtype(sensor_counts['datetime']):
    raise ValueError(f"☒The datetime colums is incorrectly being stored as a: {sensor_counts['datetime'].dtype}")
else:
    print("✅Datetime column is a datetime object")

✅Datetime column is a datetime object


While we're doing time-related things, lets create a columns to represent the month and day of week as a number. This may be more useful later than the text representations.

This is easy in pandas because, once we have created columns that have the type `datetime`, we can use `.dt` to access date- and time-related aspects. To find out what is available through the `.dt` attribute, have alook at the [dt docs](https://pandas.pydata.org/pandas-docs/dev/reference/api/pandas.Series.dt.html) and, more usefully, follow the link to the [DatetimeIndex docs](https://pandas.pydata.org/pandas-docs/dev/reference/api/pandas.DatetimeIndex.html#pandas.DatetimeIndex). Under 'attributes' on that page it shows you all the different parts of a date that you can access. For example, `sensor_counts['datetime'].dt.month` gives you the number of the month.

<font color='orchid'> <b>Edit the code below to fill in the variables. Then you will be able to print out the year, the day of the week, and the quarter for the very first row in our dataframe.</b></font>

In [9]:
first_row = sensor_counts.loc[[0], :]  # Use loc to access the first row

year = first_row['datetime'].dt.year
# Do the same as above to create variables called 'day_of_week' and 'quarter' variables
day_of_week = first_row['datetime'].dt.day_of_week
quarter = first_row['datetime'].dt.quarter

print("The date of the first row is:", first_row['datetime'].values)
print("The year, day of week and quarter are:", year.values, day_of_week.values, quarter.values)

The date of the first row is: ['2019-11-01T17:00:00.000000000']
The year, day of week and quarter are: [2019] [4] [4]


_Two things to note about the code above for those who are interested_

1. Did you see that when we created `first_row`, we used an extra set of square brackets around the 0: `first_row = sensor_counts.loc[[0], :]`? If we didn't do this then pandas would realise that we only wanted one row, and instead of returning a DataFrame object it would return a Series. Series behave slightly differently and means the example wouldn't have worked. So adding the square brackets around the zero (`[0]`) means that we we are giving `loc` a list, rather than a single number. This 'tricks' `loc` into returning a DataFrame (even though that DataFrame only contains one row).

2. In the `print` statement we use `.values`. The is because `.dt` returns a series, and when we `print` that we also get things like the row number, which makes the print statement messy. `.values` means we _only_ get the value, so it looks neater.

3. Question for those who really want to understand what's going on: why are our year, day and quarter values surrounded by square brackets when they are printed out?

Use `dt` to create columns to store the weekday and months as numbers

In [10]:
sensor_counts['weekday_num'] = sensor_counts['datetime'].dt.weekday +1  # Add 1 otherwise the days would run from 0 to 6
sensor_counts['month_num'] = sensor_counts['datetime'].dt.month

It will also be useful later to have a datetime column that just has the day, not the full minutes and hours. This is so that we can join it to other datasets that have daily resolution (don't worry about this - ask someone if you're not sure).

In [11]:
sensor_counts['datetime_day'] = sensor_counts['datetime'].dt.floor('D')

There is one other date-related thing we need to do, related to **cyclical features**. Recall that in the lecture we discussed that times and days have big jumps (e.g. at midnight, from Sunday -> Monday and from December -> Janurary). To get round this, we create _sine_ and _cosine_ transformations of those variables. Then the model is able to use two columns to capture each feature wihtout the big jumps.

<font color='orchid'> <b>Run the code below to create new features for the time, month and weekday.</b></font>  Don't worry about how this works, but if you would like to know please ask someone and we can explain it -- the concept isn't hard but the code includes some features that you haven't seen yet.

In [14]:
def add_sin_and_cos_features(df, column_to_transform):
    df['Sin_{}'.format(column_to_transform)] = np.sin(2 * np.pi * df[column_to_transform] / max(df[column_to_transform])) 
    df['Cos_{}'.format(column_to_transform)] = np.cos(2 * np.pi * df[column_to_transform] / max(df[column_to_transform]))
    return df

for col in ['time', 'month_num', 'weekday_num']:
    sensor_counts = add_sin_and_cos_features(sensor_counts, col)

In [15]:
sensor_counts

,datetime,year,month,mdate,day,time,sensor_id,hourly_counts,weekday_num,month_num,datetime_day,Sin_time,Cos_time,Sin_month_num,Cos_month_num,Sin_weekday_num,Cos_weekday_num
0,2019-11-01 17:00:00,2019,November,1,Friday,17,34,300,5,11,2019-11-01,-0.997669,-0.068242,-0.500000,8.660254e-01,-0.974928,-0.222521
1,2019-11-01 17:00:00,2019,November,1,Friday,17,39,604,5,11,2019-11-01,-0.997669,-0.068242,-0.500000,8.660254e-01,-0.974928,-0.222521
2,2019-11-01 17:00:00,2019,November,1,Friday,17,37,216,5,11,2019-11-01,-0.997669,-0.068242,-0.500000,8.660254e-01,-0.974928,-0.222521
3,2019-11-01 17:00:00,2019,November,1,Friday,17,40,627,5,11,2019-11-01,-0.997669,-0.068242,-0.500000,8.660254e-01,-0.974928,-0.222521
4,2019-11-01 17:00:00,2019,November,1,Friday,17,36,774,5,11,2019-11-01,-0.997669,-0.068242,-0.500000,8.660254e-01,-0.974928,-0.222521
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5480637,2024-03-22 15:00:00,2024,March,22,Friday,15,45,1388,5,3,2024-03-22,-0.816970,-0.576680,1.000000,6.123234e-17,-0.974928,-0.222521
5480638,2023-05-27 13:00:00,2023,May,27,Saturday,13,63,1052,6,5,2023-05-27,-0.398401,-0.917211,0.500000,-8.660254e-01,-0.781831,0.623490
5480639,2024-03-23 16:00:00,2024,March,23,Saturday,16,50,557,6,3,2024-03-23,-0.942261,-0.334880,1.000000,6.123234e-17,-0.781831,0.623490
5480640,2024-04-17 16:00:00,2024,April,17,Wednesday,16,20,478,3,4,2024-04-17,-0.942261,-0.334880,0.866025,-5.000000e-01,0.433884,-0.900969


[ ] TODO Check for duplicates and other errors

### Sensor locations

We have now finished reading and preparing the footfall counts. 

We know what the counts at each sensor are, but we don't know _where_ the sensors are located. We need to know this so that we can map the sensor counts, and also so that we can join the sensors to other spatial data about the local built environment. The sensor locations are a separate file that we donloaded from the Melbourne Data Portal. For conveinence we have put the file on module repository so you can load it easily:

In [16]:
sensor_locations = pd.read_csv("https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/sensor_locations.csv")
sensor_locations

,Unnamed: 0,sensor_id,Name,sensor_name,installation_date,status,note,Latitude,Longitude,location,Note,Location_Type,Status
0,0,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.965210,"(-37.81573422, 144.96521044)",NaN,NaN,NaN
1,1,50,Faraday St-Lygon St (West),Lyg309_T,30/11/2017,A,NaN,-37.798082,144.967210,"(-37.79808191, 144.96721014)",NaN,NaN,NaN
2,2,73,Bourke St - Spencer St (South),Bou655_T,02/10/2020,I,NaN,-37.816957,144.954154,"(-37.81695684, 144.95415373)",NaN,NaN,NaN
3,3,66,State Library - New,QVN_T,06/04/2020,A,NaN,-37.810578,144.964443,"(-37.81057845, 144.96444294)",NaN,NaN,NaN
4,4,59,Building 80 RMIT,RMIT_T,13/02/2019,A,NaN,-37.808256,144.963049,"(-37.80825648, 144.96304859)",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...
137,128,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.961860,"(-37.82590962, 144.96185972)",NaN,Outdoor,A
138,129,150,narrm ngarrgu Library - Level 1 Main Stairs B,narrLibL1MB_T,2023-10-23,NaN,NaN,-37.807912,144.958201,"(-37.80791198, 144.95820087)",NaN,Indoor,A
139,130,152,narrm ngarrgu Library - Level 2 - Study Area L...,narrLibL2S1_T,2023-10-23,NaN,NaN,-37.807767,144.958440,"(-37.80776728, 144.95843977)",NaN,Indoor,A
140,131,154,narrm ngarrgu Library - Level 3 Children's Lib...,narrLibL3C1_T,2023-10-23,NaN,NaN,-37.807784,144.958628,"(-37.80778437, 144.95862772)",NaN,Indoor,A


Again, you need to get rid of that strange column.
    
<font color='orchid'> <b>Write the code below remove the column called 'Unnamed: 0'.</b></font>.

In [17]:
sensor_locations = sensor_locations.drop(columns = ['Unnamed: 0'])

### Merge the sensor counts with the sensor locations

Now that we have counts for the sensors as well as their locations, we can merge the two DatFrames. This is made easy because both datasets have a column called `sensor_id`, which uniquely identifies a sensor. 

We will use the `pd.merge` function to merge them. Have a quick look at the [pandas.merge() documentation](https://pandas.pydata.org/docs/reference/api/pandas.merge.html). The function has two _requred_ parameters: 'left' and 'right'. These are the two datasets that you want to merge. In our case it also needs the 'on' parameter so that it knows which column to use to merge the datasets. For example, if we wanted to merge two datasets, `df_a` and `df_b`, using column `id` as the linking column we would use `merge` like this:
```python
merged_df = pd.merge(df_a, df_b, on='id')
```
<font color='orchid'> <b>Edit the code below to create a new variable called `location_counts` by merging the `sensor_locations` and `sensor_counts` dataframes using the `sensor_id` column</b></font>.

In [18]:
location_counts = pd.merge(sensor_locations, sensor_counts, on='sensor_id')
location_counts

,sensor_id,Name,sensor_name,installation_date,status,note,Latitude,Longitude,location,Note,...,hourly_counts,weekday_num,month_num,datetime_day,Sin_time,Cos_time,Sin_month_num,Cos_month_num,Sin_weekday_num,Cos_weekday_num
0,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,...,744,6,1,2011-01-01,0.000000,1.000000,0.500000,8.660254e-01,-7.818315e-01,0.623490
1,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,...,444,6,1,2011-01-01,0.269797,0.962917,0.500000,8.660254e-01,-7.818315e-01,0.623490
2,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,...,263,6,1,2011-01-01,0.519584,0.854419,0.500000,8.660254e-01,-7.818315e-01,0.623490
3,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,...,169,6,1,2011-01-01,0.730836,0.682553,0.500000,8.660254e-01,-7.818315e-01,0.623490
4,16,Australia on Collins,Col270_T,30/03/2009,R,Device moved to location ID 53 (22/09/2015),-37.815734,144.96521,"(-37.81573422, 144.96521044)",NaN,...,79,6,1,2011-01-01,0.887885,0.460065,0.500000,8.660254e-01,-7.818315e-01,0.623490
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5480637,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,...,160,2,4,2024-04-02,-0.816970,-0.576680,0.866025,-5.000000e-01,9.749279e-01,-0.222521
5480638,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,...,149,2,9,2024-09-03,-0.136167,-0.990686,-1.000000,-1.836970e-16,9.749279e-01,-0.222521
5480639,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,...,3,4,5,2024-05-23,0.979084,0.203456,0.500000,-8.660254e-01,-4.338837e-01,-0.900969
5480640,140,COM Pole 2837 - Boyd Park,Boyd2837_T,2023-11-02,NaN,NaN,-37.825910,144.96186,"(-37.82590962, 144.96185972)",NaN,...,5,4,5,2024-05-30,0.979084,0.203456,0.500000,-8.660254e-01,-4.338837e-01,-0.900969


To make sure this worked, lets check that your new `location_counts` dataframe has the same number of rows as the original `sensor_counts` dataframe

In [19]:
if len(location_counts) == len(sensor_counts):
    print("Merged dataframe has the correct number of rows")
else:
    print("Merged dataframe length", len(location_counts), "doesn't match the original sensor counts length", len(sensor_counts))


Merged dataframe has the correct number of rows


## Weather and holiday data

We need information about when school and public holidays take place, and also information about the weather. These were collected from a few places, see [the paper](https://doi.org/10.1177/23998083251319058) if you're interested. 

One thing to note about the code below is that we use the `parse_dates=['datetime']` argument. This tries to automatically create a datetime object when reading the 'datetime' column, rather than storing it as a string. It means that we don't need to convert the column later (like we had to do for the sensor data). The `dayfirst` argument tells pandas that for two of our files the day comes first, not the year.  (We could have used this feature when reading the sensor data earlier, but it was good to see how you have to do it manually).

### Load weather and holiday data

In [46]:
url_to_data = "https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/"

daily_rainfall = pd.read_csv(url_to_data+"DailyRainfallData.csv", parse_dates=['date'])
public_holidays = pd.read_csv(url_to_data+"publicholidays.csv", parse_dates=['date'])
school_holidays = pd.read_csv(url_to_data+"schoolholidays.csv", parse_dates=['date'], dayfirst=True)
# The second weather data file we need is quite big, so it has been compressed with gzip and we need to uncompress it
weather_data = pd.read_csv(url_to_data+"weather_data_allyears.csv.gz", parse_dates=['datetime'], dayfirst=True, compression='gzip')

<font color='orchid'> <b>Edit the code below to have a look at those four dataframes that you just loaded. Do they _look_ OK? Are the data stored in the right format? Are there any duplicate rows? Etc.</b></font>.

In [21]:
# Check that the data are OK in this cell. If you want to add new cells (e.g. one for each dataframe) you can do




### Merge the weather and holiday data with the sensor data

We want to attach these new dataframes to our sensor data. All of our dataframes have a column called `date`, so we can merge by comparing that column to our `datetime_day`.

**IMPORTANT!**. There may be some specific dates and/or times missing from our weather and holiday data. The default merge behaviour is to drop rows where there is no match. We can tell pandas not to do this by making it a 'left' join. That means keep all the rows in the 'left' dataset, and if it cannot find a matching value then insert 'na' into that row. We can use the `how="left"` argument to `merge()` and make sure that `location_counts` is the first dataset we pass (that one is treated as the 'left' dataset, the other is treated as the 'right' one).

<font color='orchid'> <b>Run the chunk below to see how to attach the daily_rainfall data to the sensor data</b></font>.

In [48]:
location_counts = pd.merge(location_counts, daily_rainfall, left_on="datetime_day", right_on="date", how="left")

<font color='orchid'> <b>Now edit the chunks below to merge the other three datasets to the sensors data</b></font>. 

Hint: once we ran the code `location_counts = pd.merge(location_counts, daily_rainfall, on="datetime")` we created a new `locations_counts` dataframe and deleted the old one. This means that if we had done something wrong with the merge, we would have to go back and re-read the sensor data, re-merge it to the locations, etc. It's not a problem, but it can be annoying, especially if some of the earlier merge operations took a long time. So, if you want to check whether the merge looks OK before doing it, you can just call the `pd.merge()` function but not assign the resulting new dataframe to the `location_counts` variable. For example you could do this:
```python
temporary_df = pd.merge(location_counts, daily_rainfall, on="datetime")
```
and then analyse `temporary_df` to see if the merge worked first. Or, if you just do: 
```python
pd.merge(location_counts, daily_rainfall, on="datetime")
```
pandas will helpfully show you what the result of the merge will look like (although it doesn't save the result, so you can'd to any further analysis)

In [ ]:
# Merge public_holidays




In [ ]:
# Merge school_holidays




In [ ]:
# Merge weather_data




In [ ]:
# REMOVE FROM STUDENT COPY
for df in [public_holidays, school_holidays, weather_data]:
    location_counts = pd.merge(location_counts, df, on="datetime", how="left")
location_counts

Now use `.info()` to check that the columns in the new `location_counts` dataframe look OK. Do you have the right columns and are they of the expected type?

In [ ]:
location_counts.info()

### Check the merges worked OK

Just because the columns exist in the merged dataframe doesn't mean that the merge actually worked correctly. There may be some rows where pandas couldn't find a match.

A nice easy way to check whether the merge was OK is to see whether any 'na' values were introduced after the merge. This happens when there are some rows in one dateset that pandas can't find a partner for in the other data.

The following will display a table that contains all rows from `location_counts` that have an `na` value in any of the columns 'rainfall_mm', 'public_holiday', 'school_holiday' and 'Temp'. If this resulting table has no rows in it, then it means that tehre are no rows in `location_counts` with missing data in those columns (i.e. the merge worked!).

<font color='orchid'> <b>Run the code below. Are there any rows with missing data?</b></font>

In [ ]:
location_counts[location_counts[['rainfall_mm', 'public_holiday', 'school_holiday', 'Temp']].isna().any(axis=1)]

## Load and merge the built-environment spatial data 

Now we want to join in other data that may influence pedestrian footfall. Things like hospitals, universities, train stations, etc., etc.

Preparing the spatial data quite laborious. We downloaded OpenStreetMap data and then used the spatial analysis functions available in geopandas to find the number of different tyles of objects (like trees, stations, bike stands, bus stops, etc.) within a certain radius of each sensor. If you are interested in how we did this have a look at the [FindFeaturesInProximityToSensors.ipynb](https://github.com/nickmalleson/footfall/blob/main/MelbourneAnalysis/1.%20PreparingData/2.%20LinkSpatialFeaturesDataToSensors/FindFeaturesInProximityToSensors.ipynb) script in the paper's repository. 

For this practical, we will just load the pre-processed data and join it to the sensors. The data that we will read is a table that has, for each sensor, the numbers of different types of physical features within a certain distance (400m in our case)

In [ ]:
num_features_near_sensors = pd.read_csv("https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/refs/heads/main/data/week_7/num_features_near_sensors_400.csv")
num_features_near_sensors

<font color='orchid'> <b>Write the code below to merge the `location_counts` data frame with the `num_features_near_sensors` dataframe, using the `sensor_id` column that is common to both dataframes.</b></font>.

In [ ]:

location_counts = pd.merge(location_counts, num_features_near_sensors, on="sensor_id")


In [ ]:
# This checks you did the merge correctly by checking that the 'trees' column exists and isn't NA
# 'trees' is just one of the columns from the num_features_near_sensors data, we could have checked any of them
if not "trees" in location_counts.columns:
    raise ValueError("There is no 'trees' column after merging, something has gone wrong.")
else:
    print("There is a 'trees' column after the merge")

# If we get here then no error was raised so presumably the trees column exists. Just check it is not NA
if location_counts['trees'].isna().all():
    raise ValueError("'trees' column exists but all values are NA — the merge may have failed.")
else:
    print("Merge successful: 'trees' column found and contains valid data.")

## Some final cleaning steps

Here are some final steps that we need to do to prepare the data for anlaysis later. We didn't always know that these were necessary at first, so often we would do some anlaysis, find out something didn't work, and come back here to do some more data preparation.

The chunks below are mostly self explanatory, <font color='orchid'> <b>run the chunks below but check you know what they are doing.</b></font>.

# Data Analysis

[ ] TODO Ask students to build on VIS lecture to make some graphs etc. to show some interesting features (look at how footfall has changed over time, do some maps of features, etc.)

# Modelling

Now we are ready to do some modelling. Recall from the lecture that we need to do three things:

1. **Model Selection** to chose the best _type_ of model
2. **Hyperparameter tuning** to find the best parameters for our chosen type of model
3. **Model evaluation** to see how well our optimised model works on data that it hasn't seen before.

![Model building pipeline](https://github.com/MSc-Urban-Environmental-Leeds/GEOG5415M-Programming-for-Spatial-Data-Science/raw/main/notebooks/screenshots/week_7-ModelBuildingPipeline.png)

## Data Preparation

Before continuing, lets get our _X_ (predictors) and _y_ (target) datasets ready. This mostly consists of just choosing the rows and columns in the `location_counts` data frame that we want to use for modelling. 

Lets start by selecting the rows that we want. Remember that each row holds the footfall count for a particular hour in our study period. During and after COVID, footfall patterns changed dramatically, so it is unlikely that a model will be able to predict footfall under _normal_ conditions as well as during the COVID period. Therefore we will remove the COVID period from our training data.

Also remember that there is a convention to call our data _X_ and _y_, so we will do that

Start with the _y_ data (our target). Use `loc` to filter out all rows that were before 2020 (the rows) and select only the 'hourly_counts' column.

As a reminder, the syntax of `loc` is like this:
```python
  df.loc[ <rows_to_filter>, <columns_to_select> ]
```

In [ ]:
y = location_counts.loc[ location_counts['datetime'] < '2020', 'hourly_counts']

Now do the same for the _X_ dataset (our predictors). This is slightly more complicated though because rather than just selection one column (the 'hourly_counts') we want to select all the columns that we think might be related to footfall. Lets define the columns we want in their own list to make things easier. There is no 'correct' list here. If you're interested you could come back and add or remove variables from this list and see how it affects the model (remember that `location_counts.columns` will show you all the colums that are available).

In [ ]:
# Choose the columns we want to use as our predictor variables, making a big list
predictor_columns = [
    # The time-related variables:
    'Sin_time', 'Cos_time', 
    'Sin_month_num', 'Cos_month_num', 
    'Sin_weekday_num', 'Cos_weekday_num',
    # Weather-related:
    'rainfall_mm', 'Temp', 'Humidity', 'Pressure','WindSpeed',
    # Holidays:
    'public_holiday', 'school_holiday', 
    # Built-environment
    'lights', 'street_inf', 'bikes', 'landmarks', 'memorials', 'trees', 
    'transport_stops', 'bus-stops', 'tram-stops', 'metro-stations', 'taxi-ranks',
    'big-car-parks', 'buildings_2019','avg_n_floors_2019'
]

Now we can use `loc` like we did before.

<font color='orchid'> <b>Complete the code below to use the `loc` accessor to choose all rows before 2020 and the columns we defined in the `predictor_columns` list.</b></font>

In [ ]:
# Create the 'X' variable with rows before 2020 and columns from the 'predictor_columns' list

X = location_counts.loc[ location_counts['datetime'] < '2020', predictor_columns]


Have a look at the two new dataframes (_X_ and _y_). Do they have the right columns? Do they have the same number of rows?

<font color='orchid'> <b>Do a couple of checks below to see that _X_ and _y_ are as you expect.</b></font>

In [ ]:
# This just checks that the number of rows are the same
if not len(X) == len(y):
    raise ValueError("The number of rows in X and y are different")
else:
    print("X and y have the same number of rows, which is what we want")

## Model selection

Sometimes you know which kind of model will best suite your application, but other times you need to try a few different ones. In the full paper we tested three types of model:

1. **Linear regression**: our data are non-linear, so we don't expect this to work well, but it is a good 'benchmark'
2. **Random forest regression**: these handle non-linearity well
3. **XGBoost:**: a gradient boosting method that can out-perform random forest

Here, to simplify the code slightly, we will just test **Linear regression** and **Random forest regression**.

Start by creating our scikit-learn `Pipeline` objects. Note: as part of the pipeline we can also use the `StandardScaler` to scale our data using Z-Scores.

In [ ]:
# Pipeline for linear regression
lr_model_pipeline = Pipeline(steps=[
    ['scaler',StandardScaler()],             # Step 1: normalise ('scale') the data
    ['linear_regressor',LinearRegression()]  # Step 2: run the linear regression
])

# Pipeline for random forest
rf_model_pipeline = Pipeline(steps=[
    ['scaler',StandardScaler()],
    ['rf_regressor', RandomForestRegressor(n_jobs = os.cpu_count())]  # Note: njobs allows to run on more cores
])

We could do full cross-validation with model selection, but for this example we wont. Instead, we will just create our own train and validate (test) datasets, using the handy `train_test_split` function that scikit-learn provides. We need four datasets: train/test for our _X_ data, and train/test for our _y_ data.

<font color='orchid'> <b>Run the code below to have scikit-learn create test and train datasets.</b></font>

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y)

Now we try fitting the linear regression and random forest models to those data (fit it to the training dataset and then test it's accuracy on the test data)

<font color='orchid'> <b>Run the code below to fit the linear regression to the training data.</b></font>

In [ ]:
lr_model_pipeline.fit(X_train, y_train)

Now do the same for the random forest pipeline.
    
<font color='orchid'> <b>Run the code below to fit the linear regression to the training data.</b></font>

In [ ]:
# Fit the random forest just like the linear regression one above
rf_model_pipeline.fit(X_train, y_train)


We have trained our models. Lets see which one worked best by seeing how well it can predict footfall, using the data in `X_test`, which it hasn't seen before, to predict footfall in `y_test`.

We will start with the linear regression:

In [ ]:
# Create a new dataset of model predictions, giving the model the X_test data
lr_predict = lr_model_pipeline.predict(X_test)

# Now calculate two error metrics to see how well it did
lr_rmse = np.sqrt(mean_squared_error(y_test, lr_predict))
lr_r2 = r2_score(y_test, lr_predict)
print("The linear regression model got rmse and r2 scores of:", lr_rmse, lr_r2)

OK, so we have used the fitted linear regression model to make predictions and calculated how good those predictions were when compared to some hourly counts.

<font color='orchid'> <b>Fill in the code below to make predictions for the random forest model and calculate how well the model worked.</b></font>



In [ ]:
# Create a new dataset of model predictions, giving the model the X_test data
rf_predict = rf_model_pipeline.predict(X_test)

# Now calculate two error metrics to see how well it did
rf_rmse = np.sqrt(mean_squared_error(y_test, rf_predict))
rf_r2 = r2_score(y_test, rf_predict)
print("The random forest model got rmse and r2 scores of:", rf_rmse, rf_r2)

**Which model was better??** 

